In [1]:
import pandas as pd

import job_search.utils as utils
import job_search.config as conf
from job_search.config import P_RAW, P_INTERIM
from job_search.utils import now, reload
import job_search.scrape as sc
from job_search.scrape import (SITEMAP, SITEMAP_INDEX, SITEMAP_COMPANIES,
                               SITEMAP_JOB_TITLES, SITEMAP_LOCATIONS)

P_raw_date = P_RAW / now(time=False).replace('-', '/')
utils.jupyter_css_style()

In [40]:
import aw
reload(sc)

## Sitemaps

In [5]:
companies_df = sc.load_sitemap(SITEMAP_COMPANIES)
companies = companies_df['sitemap'].str.removeprefix('company/')
companies

Loading from: C:\Users\Alex\Dev\job-search\data\interim\2026\04\19\sitemap-companies-index.parquet


0                011h.com
1                  01c.ai
2        01informatica.es
3           01systems.com
4        021strategic.com
               ...       
47922           zzazz.com
47923          zzeeks.com
47924      zzf-potsdam.de
47925    zzi-melsungen.de
47926              zzr.de
Name: sitemap, Length: 197927, dtype: object

In [64]:
locations_df = sc.load_sitemap(SITEMAP_LOCATIONS)
locations = locations_df['sitemap'].str.removeprefix('jobs/locations/')
california_mask = locations.str.contains(r'\bcalifornia')
locations[california_mask]

100%|██████████| 1/1 [00:00<00:00,  1.57it/s]

Saving to: C:\Users\Alex\Dev\job-search\data\interim\2026\04\20\sitemap-locations-index.parquet


29       acalanes-ridge-california
40                acton-california
79             adelanto-california
102              agoura-california
103        agoura-hills-california
                   ...            
20976             yreka-california
20977         yuba-city-california
20978           yucaipa-california
20979      yucca-valley-california
20991           zayante-california
Name: sitemap, Length: 1224, dtype: object

In [7]:
job_titles_df = sc.load_sitemap(SITEMAP_JOB_TITLES)
job_titles = job_titles_df['sitemap'].str.removeprefix('jobs/').str.removesuffix('/locations/united-states')
data_mask = job_titles.str.contains('data')
scien_mask = job_titles.str.contains('scien')
ai_mask = job_titles.str.contains(r'\bai\b')
job_titles[ai_mask]

Loading from: C:\Users\Alex\Dev\job-search\data\interim\2026\04\19\sitemap-job-titles-index.parquet


2522     accelerated-leadership-program-rd-career-path-...
4168     accounting-manager-corporate-process-and-ai-ex...
4232               accounting-process-ai-technology-expert
10095                                advanced-ai-architect
10096                                 advanced-ai-engineer
                               ...                        
2030                  vp-of-physical-ai-autonomous-systems
2331                          vp-security-it-ai-enablement
2476                             vp-workforce-ai-solutions
6883                                    wealth-ai-engineer
7065      wealth-technology-ai-and-marketing-group-manager
Name: sitemap, Length: 3295, dtype: object

In [8]:
index_df = sc.load_sitemap(SITEMAP_INDEX)
index = index_df['sitemap'].str.removeprefix('viewjob/')
index

Loading from: C:\Users\Alex\Dev\job-search\data\interim\2026\04\19\sitemap-index.parquet


0       2tbusumgzyt38dzo
1       o2it0k611oq3692h
2       vjms85as2pau2ed9
3       smxf8atte52hhj3f
4       i2djvefiqa7ao2ze
              ...       
2193    x3l4j89l7mpoqh5w
2194    w3og0aezyn68kbm7
2195    nyr06qvlvno7ud7q
2196    2hdhi6y7tyabi4s6
2197    zp0mh7iztdy9kj9r
Name: sitemap, Length: 4152198, dtype: object

In [9]:
# index.str.len().pipe(aw.vcounts)
index[index.str.len() > 16]

25778    12fcd5aa-fb23-4e02-8813-3da58fdf4ffc
25869    3760dcf5-ff26-4392-8d7e-e43b81ab8999
29199    6c566bdc-e793-4375-a859-307bb039fe6b
34446    01e4a97b-4185-4597-8260-34a851a72cdc
41977    0a29f932-54eb-4ee9-a687-3430eab96eba
48346    50d9934e-df94-4c46-b189-24cd722783be
17771    04351cb0-6d95-47ed-b0b2-210ce72b6b24
21327    b440449b-47ac-441d-ab94-d5f7b1c1add1
38789    049b571a-56cf-4917-9011-0468f98cee27
49912    bddcf0cf-19c3-4bb4-a413-2251addd7cc4
642      a1cc2b18-78e2-487d-8f3a-38c6d47b04cc
23833    4c570253-b7aa-4652-b4bc-7c0945c2c572
44179    09c15fb5-eb4f-41d6-b7f0-037c9dfa52e2
23213    45085c69-6934-40f0-ac22-5bd6de4c92ce
37447    a6745876-daf4-4ab6-aaed-fafffe3818d5
32989    07926efd-afba-4ff7-99d2-d20f41459f1f
28960    67119ec1-eea8-4499-9026-495c9a757786
29394    bc0330c2-a404-467f-af61-c53e615d1674
24639    a662ab81-c365-475d-8623-6ec75c201639
30660    36a925b6-127d-40c9-862c-c9fb2dc2c7f3
30773    eb5d6567-1d26-4490-9548-6a2008d1b4b6
30970    ad690744-9271-4b70-9b13-3

## Scraping

In [21]:
## Takes ~15 secs
ds_job_titles = sc.load_ds_job_titles()

Loading from: C:\Users\Alex\Dev\job-search\data\interim\2026\04\20\sitemap-job-titles-index.parquet


In [63]:
ds_dict = sc.load_jobs()
ds_dict['pageProps'].keys()
# sc.load_jobs()['pageProps']['ssrPage']#.keys()

Reading: C:\Users\Alex\Dev\job-search\data\interim\2026\04\20\jobs\data-scientist\page.json.gz


dict_keys(['title', 'description', 'jobTitle', 'jobTitleText', 'locationCode', 'locationName', 'locationKeywords', 'initialSearchState', 'ssrHits', 'ssrPage', 'ssrTotalCount', 'ssrCompanyCount', 'ssrPageSize', 'ssrIsLastPage', 'ssrError', 'canonicalUrl', 'prevPageUrl', 'nextPageUrl', 'nearbyCities', 'ssrTimings'])

In [93]:
P_EXTERNAL = conf.P_EXTERNAL

def load_cities():
    cities_df = pd.read_csv(P_EXTERNAL / 'City_Boundaries.csv')
    return cities_df

cities = pd.read_csv(P_EXTERNAL / "cities.csv", header=None)[0]
bay_cities = cities[~cities.str.startswith("#")].reset_index(drop=True)
bay_cities


0      San Francisco
1            Alameda
2             Albany
3           Berkeley
4             Dublin
           ...      
96      Rohnert Park
97        Santa Rosa
98        Sebastopol
99            Sonoma
100          Windsor
Name: 0, Length: 101, dtype: object

In [95]:
cities_df = load_cities()
ca_cities = cities_df['CITY'].str.lower().str.replace(' ', '-')
ca_cities

0          adelanto
1      agoura-hills
2           alameda
3            albany
4          alhambra
           ...     
477      yountville
478           yreka
479       yuba-city
480         yucaipa
481    yucca-valley
Name: CITY, Length: 482, dtype: object

In [97]:
# set(ca_locations) - set(ca_cities)

In [83]:
ca_locations = locations[california_mask].str.removesuffix('-california')
ca_locations

29       acalanes-ridge
40                acton
79             adelanto
102              agoura
103        agoura-hills
              ...      
20976             yreka
20977         yuba-city
20978           yucaipa
20979      yucca-valley
20991           zayante
Name: sitemap, Length: 1224, dtype: object

In [120]:
reload(sc)

In [121]:
sc.load_ds_ca()

Reading: C:\Users\Alex\Dev\job-search\data\interim\2026\04\20\ds-ca-2tzfqdib\page0.json.gz
Saving: C:\Users\Alex\Dev\job-search\data\interim\2026\04\20\ds-ca-2tzfqdib\page1.json.gz
Saving: C:\Users\Alex\Dev\job-search\data\interim\2026\04\20\ds-ca-2tzfqdib\page2.json.gz
Saving: C:\Users\Alex\Dev\job-search\data\interim\2026\04\20\ds-ca-2tzfqdib\page3.json.gz
Saving: C:\Users\Alex\Dev\job-search\data\interim\2026\04\20\ds-ca-2tzfqdib\page4.json.gz
Saving: C:\Users\Alex\Dev\job-search\data\interim\2026\04\20\ds-ca-2tzfqdib\page5.json.gz
Saving: C:\Users\Alex\Dev\job-search\data\interim\2026\04\20\ds-ca-2tzfqdib\page6.json.gz
Saving: C:\Users\Alex\Dev\job-search\data\interim\2026\04\20\ds-ca-2tzfqdib\page7.json.gz
Saving: C:\Users\Alex\Dev\job-search\data\interim\2026\04\20\ds-ca-2tzfqdib\page8.json.gz
Saving: C:\Users\Alex\Dev\job-search\data\interim\2026\04\20\ds-ca-2tzfqdib\page9.json.gz
Saving: C:\Users\Alex\Dev\job-search\data\interim\2026\04\20\ds-ca-2tzfqdib\page10.json.gz
Saving: 

False